In [2]:
# Cài gdown để download từ Google Drive
!pip install gdown -q

# Download dataset OCR (YOLO format)
!gdown 1Mdtfn39Jt53u9Y81jhoM-7pdQT7B_dF6 -O /kaggle/working/ocr_dataset.zip

# Giải nén
!unzip -q /kaggle/working/ocr_dataset.zip -d /kaggle/working/ocr_dataset

# Xem cấu trúc
import os
for root, dirs, files in os.walk('/kaggle/working/ocr_dataset'):
    level = root.replace('/kaggle/working/ocr_dataset', '').count(os.sep)
    if level < 3:
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level == 2:
            print(f'{indent}  ({len(files)} files)')

Downloading...
From (original): https://drive.google.com/uc?id=1Mdtfn39Jt53u9Y81jhoM-7pdQT7B_dF6
From (redirected): https://drive.google.com/uc?id=1Mdtfn39Jt53u9Y81jhoM-7pdQT7B_dF6&confirm=t&uuid=63db32af-970a-4cbd-aa6e-951beed2a111
To: /kaggle/working/ocr_dataset.zip
100%|████████████████████████████████████████| 124M/124M [00:02<00:00, 54.6MB/s]
ocr_dataset/
  yolo_plate_ocr_dataset/


In [3]:
# Xem 1 label mẫu để biết format
import os

label_dir = None
for root, dirs, files in os.walk('/kaggle/working/ocr_dataset'):
    for f in files:
        if f.endswith('.txt') and f != 'classes.txt':
            label_dir = root
            sample = f
            break
    if label_dir:
        break

with open(f'{label_dir}/{sample}') as f:
    content = f.read()
    print(f"Sample label ({sample}):")
    print(content)
    cols = len(content.strip().split('\n')[0].split())
    print(f"\nSố cột: {cols}")
    print("Format: YOLO detection (5 cột)" if cols == 5 else f"Format khác ({cols} cột)")

# Xem classes
for root, dirs, files in os.walk('/kaggle/working/ocr_dataset'):
    if 'classes.txt' in files:
        with open(f'{root}/classes.txt') as f:
            classes = f.read()
        print(f"\nClasses:\n{classes}")
        break

Sample label (iwt72.txt):
11 0.181818 0.580986 0.082111 0.443662
35 0.269795 0.559859 0.082111 0.457746
21 0.354839 0.549296 0.082111 0.436620
4 0.533724 0.500000 0.076246 0.436620
0 0.615836 0.478873 0.070381 0.436620
8 0.737537 0.450704 0.073314 0.436620
5 0.818182 0.433099 0.082111 0.457746


Số cột: 5
Format: YOLO detection (5 cột)


In [4]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.1 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you 

In [10]:
# Đếm chính xác từ label files
import glob

all_cls = set()
for fpath in glob.glob('/kaggle/working/ocr_char/labels/train/*.txt'):
    with open(fpath) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                all_cls.add(int(parts[0]))

max_cls = max(all_cls)
NC      = max_cls + 1
print(f"Max class index: {max_cls}")
print(f"NC = {NC}")
print(f"All classes: {sorted(all_cls)}")

Max class index: 35
NC = 36
All classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]


In [9]:
CLASSES = [
    '0','1','2','3','4','5','6','7','8','9',  # 0-9  (10)
    ':',                                        # 10   (1)
    'A','B','C','D','E','F','G','H','I','J',   # 11-20 (10)
    'K','L','M','N','O','P','Q','R','S','T',   # 21-30 (10)
    'U','V','W','X','Y','Z',                   # 31-35 (5... = 36 tổng)
]

print(f"len = {len(CLASSES)}")  # → 36 ✅

NC   = len(CLASSES)
data = {
    'train' : f'{out}/images/train',
    'val'   : f'{out}/images/val',
    'nc'    : NC,
    'names' : CLASSES
}
with open('/kaggle/working/data_ocr.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False, allow_unicode=True)

print(open('/kaggle/working/data_ocr.yaml').read())

len = 37
names:
- '0'
- '1'
- '2'
- '3'
- '4'
- '5'
- '6'
- '7'
- '8'
- '9'
- ':'
- A
- B
- C
- D
- E
- F
- G
- H
- I
- J
- K
- L
- M
- N
- O
- P
- Q
- R
- S
- T
- U
- V
- W
- X
- Y
- Z
nc: 37
train: /kaggle/working/ocr_char/images/train
val: /kaggle/working/ocr_char/images/val



In [11]:
CLASSES = [
    '0','1','2','3','4','5','6','7','8','9',  # 0-9
    ':',                                        # 10
    'A','B','C','D','E','F','G','H','I','J',   # 11-20
    'K','L','M','N','O','P','Q','R','S','T',   # 21-30
    'U','V','W','X','Y','Z',                   # 31-35
]

print(f"len = {len(CLASSES)}")  # → 36 ✅

data = {
    'train' : f'{out}/images/train',
    'val'   : f'{out}/images/val',
    'nc'    : len(CLASSES),
    'names' : CLASSES
}
with open('/kaggle/working/data_ocr.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False, allow_unicode=True)

# Verify trước khi train
content = open('/kaggle/working/data_ocr.yaml').read()
print(content)

# Train
from ultralytics import YOLO
model   = YOLO('yolov8s.pt')
results = model.train(
    data     = '/kaggle/working/data_ocr.yaml',
    epochs   = 100,
    imgsz    = 320,
    batch    = 64,
    device   = 0,
    project  = '/kaggle/working/runs',
    name     = 'ocr_char_det',
    patience = 15,
    save     = True,
    plots    = True
)
print(f"\n✅ Train xong!")
for k, v in results.results_dict.items():
    print(f"  {k}: {v:.4f}")

len = 37
names:
- '0'
- '1'
- '2'
- '3'
- '4'
- '5'
- '6'
- '7'
- '8'
- '9'
- ':'
- A
- B
- C
- D
- E
- F
- G
- H
- I
- J
- K
- L
- M
- N
- O
- P
- Q
- R
- S
- T
- U
- V
- W
- X
- Y
- Z
nc: 37
train: /kaggle/working/ocr_char/images/train
val: /kaggle/working/ocr_char/images/val

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_ocr.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, ke

In [12]:
import shutil

# Copy weights ra output để download
shutil.copy(
    '/kaggle/working/runs/ocr_char_det/weights/best.pt',
    '/kaggle/working/char_recognition.pt'
)

print("✅ Sẵn sàng download!")
print("→ Vào tab Output → tìm char_recognition.pt → nhấn Download")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/runs/ocr_char_det/weights/best.pt'